In [4]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

RESULTS_ROOT = "/repo/results/metrics/final_results"
FIGS_ROOT = "/repo/results/figs"

## Get test results
MODEL_NAME_READABLE2STD = {
    "Train on Original": "baseline_train_on_original",
    "Train on Uniform Noise": "baseline_train_on_Uniform_Noise",
    r"Train on Rotated 90$\degree$": "baseline_train_on_Rotate_90deg",
    "Train on Ring Artifact": "baseline_train_on_ring_data",
    r"Train on all but Rotated 90$\degree$": "augmentation_train_exclude_Rotate_90deg",
    "Train on all but Ring Artifact": "augmentation_train_exclude_ring_data",
    # r"DANN ($\mathcal{S}=\text{Original}, \mathcal{T}=\text{Rotated 90}\degree$)": "dann_target_domain_Rotate_90deg",
    r"DANN ($\mathcal{S}=\text{Original}, \mathcal{T}=\text{Ring Artifact}$)": "dann_target_domain_ring_data",
}

METRIC_NAME_READABLE2COL = {
    "Test Accuracy (Labels)": {"eval_accuracy_branch1", "eval_accuracy"},
    "Test Macro Precision (Labels)": {"eval_precision_branch1", "eval_precision"},
    "Test Macro Recall (Labels)": {"eval_recall_branch1", "eval_recall"},
    "Test Macro F1 (Labels)": {"eval_f1_branch1", "eval_f1"},
}

TESTSET_NAME_PUBLISHED2INDEX = {
    "Original": "original",
    "Uniform Noise": "Uniform_Noise",
    r"Rotated 90$\degree$": "Rotate_90deg",
    "Ring Artifact": "ring_data",
}

FONTSIZE_L=23
FONTSIZE_S=18

In [9]:
def get_key_from_valueset(d: dict[str, set[str]], v: str):
    for k in d.keys():
        if v in d[k]:
            return k
    return None

def reverse_dict_of_set(d: dict[str, set[str]]) -> dict[str, str]:
    r = {}
    for k, vset in d.items():
        for v in vset:
            r[v] = k
    return r

def reverse_dict(d: dict[str, str]) -> dict[str, str]:
    r = {}
    for k, v in d.items():
        r[v] = k
    return r

def mean_std_dfs_to_heatmap(mean_df, std_df, ttl, vmin=0, vmax=1, figsize=None):
    if figsize==None:
        figsize=(10,6)

    plusminus_df = mean_df.map(lambda m: f"{m:.3f}") + "\n±" + std_df.map(lambda s: f"{s:.3f}")
    
    # Plot heatmap
    plt.figure(figsize=figsize)
    ax = sns.heatmap(
        mean_df,
        cmap="Blues",
        annot=plusminus_df,
        fmt="", # Keep format from above
        linewidths=0.5,
        vmin=vmin,
        vmax=vmax,
        annot_kws={"size": FONTSIZE_S}
    )
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=FONTSIZE_S)

    plt.title(ttl, pad=20, fontsize=FONTSIZE_L)
    plt.xlabel("Model", fontsize=FONTSIZE_L)
    plt.ylabel("Test Data Distortion", fontsize=FONTSIZE_L)
    plt.xticks(rotation=30, ha='right', fontsize=FONTSIZE_L)
    plt.yticks(rotation=0, fontsize=FONTSIZE_L)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGS_ROOT, ttl + ".png"))
    plt.show()

In [16]:
def create_kfold_test_matrices(figsize: tuple[int, int]):
    # Collect all relevant metrics from files
    metric_tables = {}
    for stat in ["mean", "std"]:
        metric_tables[stat] = {}
        for metric_name_readable in METRIC_NAME_READABLE2COL.keys():
            metric_tables[stat][metric_name_readable] = {}
            for model_name_readable in MODEL_NAME_READABLE2STD.keys():
                # Find a folder with the test results
                model_name_std = MODEL_NAME_READABLE2STD[model_name_readable]
                for testset_name_published in TESTSET_NAME_PUBLISHED2INDEX.keys():
                    testset_name_index = TESTSET_NAME_PUBLISHED2INDEX[testset_name_published]
                    testset_folder = os.path.join(RESULTS_ROOT,model_name_std, testset_name_index)
                    fold_test_result_fps = [os.path.join(testset_folder, x, "test_metrics.csv") for x in os.listdir(testset_folder) if os.path.isdir(os.path.join(testset_folder, x))]
                    
                    assert len(fold_test_result_fps) == 5, "Expecting 5 folds"
                
                    fold_metrics = {}
                    for fold_num in range(len(fold_test_result_fps)):
                        fold_test_result_fp = fold_test_result_fps[fold_num]
                        df = pd.read_csv(fold_test_result_fp)
                        df = df.rename(columns=reverse_dict_of_set(METRIC_NAME_READABLE2COL))
                        fold_metrics[fold_num] = df[metric_name_readable].item()
                        print(fold_metrics)
                        return
        
                # df = pd.DataFrame(fold_metrics)
    #             if stat == "mean":
    #                 metric_tables[stat][metric_name_readable][model_name_readable] = df.mean(axis=1)
    #             elif stat == "std":
    #                 metric_tables[stat][metric_name_readable][model_name_readable] = df.std(axis=1)
    #             else:
    #                 assert False

    # dfs = {}
    # for stat in ["mean", "std"]:
    #     dfs[stat] = {}
    #     for metric_name_readable, metric_dict in metric_tables[stat].items():
    #         df = pd.DataFrame(metric_dict)
    #         dfs[stat][metric_name_readable] = df
    
    # for metric_name_readable in METRIC_NAME_READABLE2COL.keys():
    #     mean_df = dfs["mean"][metric_name_readable]
    #     std_df = dfs["std"][metric_name_readable]
    #     # if "Loss" in metric_name_readable:
    #     #     mean_std_dfs_to_heatmap(mean_df, std_df, metric_name_readable, vmin=None, vmax=None)
    #     # elif "Domain" in metric_name_readable:
    #     #     mean_std_dfs_to_heatmap(mean_df.dropna(axis=1, how='any'), std_df.dropna(axis=1, how='any'), metric, figsize=(4.5,6))
    #     # else:
    #     mean_std_dfs_to_heatmap(mean_df, std_df, metric_name_readable, figsize=figsize)

test_results = create_kfold_test_matrices((14,9))

{0: 0.9997076878105816}
